# Analyzing Distortion in LMArena Setting

## Set-up

In [ ]:
from datasets import load_dataset

ds = load_dataset("lmarena-ai/arena-human-preference-140k")

In [ ]:
train_data = ds['train']

Import the utils that I wrote.

In [ ]:
import utils_3 as ut

## Filtering Data via Vectors

In [ ]:
ds_keys = [
    ('is_code',),
    ('category_tag', 'creative_writing_v0.1', 'creative_writing'),
    ('category_tag', 'criteria_v0.1', 'complexity'),
    ('category_tag', 'criteria_v0.1', 'creativity'),
    ('category_tag', 'criteria_v0.1', 'domain_knowledge'),
    ('category_tag', 'criteria_v0.1', 'problem_solving'),
    ('category_tag', 'criteria_v0.1', 'real_world'),
    ('category_tag', 'criteria_v0.1', 'specificity'),
    ('category_tag', 'criteria_v0.1', 'technical_accuracy'),

    ('category_tag', 'math_v0.1', 'math'),
    ('category_tag', 'if_v0.1', 'if'),
]

In [ ]:
vector_len = len(ds_keys)

In [ ]:
import numpy as np

def bit_strings(n):
    x = np.arange(2**n, dtype=np.uint32)[:, None]
    return ((x >> np.arange(n - 1, -1, -1)) & 1)

In [ ]:
vector_keys = bit_strings(vector_len)
print(vector_keys)

In [ ]:
all_candidates = set(train_data['model_a']).union(set(train_data['model_b']))
all_candidate_idxs = {cand : i for i, cand in enumerate(all_candidates)}

In [ ]:
from tqdm import tqdm
import numpy as np

In [ ]:
def process(ds_dict, mask, all_candidate_idxs):
    w = np.asarray(ds_dict["winner"])[mask]
    valid_mask = (w == "model_a") | (w == "model_b")

    w = w[valid_mask]
    a = np.asarray(ds_dict["model_a"])[mask][valid_mask]
    b = np.asarray(ds_dict["model_b"])[mask][valid_mask]

    a_idx = np.fromiter((all_candidate_idxs[x] for x in a), dtype=np.int64, count=len(a))
    b_idx = np.fromiter((all_candidate_idxs[x] for x in b), dtype=np.int64, count=len(b))

    model_a_wins = (w == "model_a")
    winners = np.where(model_a_wins, a_idx, b_idx)
    losers = np.where(model_a_wins, b_idx, a_idx)

    return winners, losers

## Computing Distortion and Misspecification Metrics

In [ ]:
n_items = len(all_candidate_idxs)
key_vector = vector_keys[5]

In [ ]:
def filter_fast(ds_dict, keys, key_vector, language="en", show_progress=True):
    if language is not None:
        mask = np.asarray(ds_dict["language"]) == language
    else:
        mask = np.ones(len(ds_dict), dtype=bool)

    iterator = zip(keys, key_vector)
    if show_progress:
        iterator = tqdm(iterator, total=len(keys))

    for path, target in iterator:
        values = ds_dict
        for k in path:
            values = values[k]
        values = np.asarray(values)
        mask &= (values == target)

    return mask

In [ ]:
print('filtering')
mask = filter_fast(train_data, ds_keys, key_vector)

In [ ]:
print('processing')
winners, losers = process(train_data, mask, all_candidate_idxs)

In [ ]:
len(losers)

In [ ]:
print('fitting')
r_hat, result = ut.fit_bradley_terry(winners, losers, n_items)

In [ ]:
len(r_hat)

In [ ]:
true_ranking = np.argsort(-r_hat)
borda_scores, ranking = ut.borda_from_population_utilities(r_hat[None])
distortion, k = ut.leaderboard_dist(ranking, true_ranking, r_hat)
misspec = ut.misspecification_error(winners, losers, r_hat)

In [ ]:
distortion

In [ ]:
misspec

## Actual Computations

In [ ]:
pop_utilities = []
absolute_subsection_sizes = []
beta = 1.0

misspec_errors = {}

for i, vector_key in tqdm(enumerate(vector_keys)):

    mask = filter_fast(train_data, ds_keys, vector_key, language=None)
    winners, losers = process(train_data, mask, all_candidate_idxs)
    r_hat, result = ut.fit_bradley_terry(winners, losers, n_items, beta=beta)

    misspec_error = ut.misspecification_error(winners, losers, r_hat, beta=beta)
    pop_utilities.append(r_hat)
    absolute_subsection_sizes.append(len(winners))

    assert len(winners) == len(losers)

    misspec_errors[i] = misspec_error

In [ ]:
pop_utilities = np.stack(pop_utilities)
print(pop_utilities.shape)

avg_utils = pop_utilities.sum(axis=0)
true_ranking = np.argsort(-avg_utils)

In [ ]:
borda_scores, ranking = ut.borda_from_population_utilities(pop_utilities, beta=beta)
distortion, k = ut.leaderboard_dist(true_ranking=true_ranking, ranking=ranking, avg_utils=avg_utils)

## Comparison with Population Utilities Scaled by Sizes